In [ ]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")

# Load category mapping
cat_df = pd.read_excel(FILE_PATH, sheet_name="Sheet1")
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))

# Load master data
master = pd.read_excel(FILE_PATH, sheet_name="Master Data ")

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time", "Tonnage"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

# Aggregate only 120T machines
records = []
for child, g in master.groupby("Child Part"):
    # Only if has 120T machine
    if not (g["Tonnage"] == 120).any():
        continue

    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    machines = list(set(normalize_machine(x) for x in g["Vertical Machines"].dropna().unique() if normalize_machine(x)))
    if not machines:
        continue

    records.append({
        "Child_Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown"),
        "Inventory": g["Inventory_25"].iloc[0]
    })

df_ml = pd.DataFrame(records)
print(f"ML-ready dataset created: {len(df_ml):,} rows (120T machines only)")
print(df_ml["Category"].value_counts())

# Save for next steps
df_ml.to_csv("120t_ml_ready.csv", index=False)
print("Saved → 120t_ml_ready.csv")

In [ ]:
import pandas as pd
import numpy as np

# Load your ML-ready dataset (from previous step)
df_ml = pd.read_csv("120t_ml_ready.csv")

SIM_DAYS = 30

# Initial state
inventory = dict(zip(df_ml["Child_Part"], df_ml["Inventory"]))
last_prod_day = {p: -SIM_DAYS for p in df_ml["Child_Part"]}  # start far back

sim_rows = []

for day in range(SIM_DAYS):
    print(f"Simulating day {day+1}/{SIM_DAYS}... ({len(sim_rows)} rows so far)", end="\r")
    
    # Daily demand variation (±20%)
    demand_factor = np.random.uniform(0.8, 1.2, len(df_ml))
    df_day = df_ml.copy()
    df_day["Day_Demand"] = df_day["Daily_Demand"] * demand_factor
    
    # Add current state features
    df_day["Day"] = day
    df_day["Inventory_Start_Day"] = df_day["Child_Part"].map(inventory).fillna(0)
    df_day["Days_Since_Last"] = day - df_day["Child_Part"].map(last_prod_day)
    df_day["Days_Since_Last"] = df_day["Days_Since_Last"].clip(lower=0)
    
    # Simplified remaining capacity (total across all machines)
    remaining_capacity = MAX_HOURS_PER_DAY * len(ALLOWED_MACHINES)
    
    # Sort parts by urgency for greedy simulation
    df_day["Urgency"] = (df_day["Net_Required"] / df_day["Daily_Demand"].clip(1)) + df_day["Days_Since_Last"] * 0.1
    df_day = df_day.sort_values("Urgency", ascending=False)
    
    produced_today = []
    
    for i, row in df_day.iterrows():
        if remaining_capacity <= 0:
            break
            
        prod_time = row["Cycle_Time_sec"] / 3600.0
        if prod_time <= 0:
            continue
            
        qty_possible = min(row["Net_Required"], remaining_capacity / prod_time)
        if qty_possible < 10:
            continue
            
        # Produce if high urgency or long time since last run
        produce = False
        if qty_possible > 50 or row["Days_Since_Last"] > 20 or row["Inventory_Start_Day"] / row["Day_Demand"] < 5:
            produce = True
            
        # Record row for EVERY part, every day
        sim_rows.append({
            "Day": day,
            "Child_Part": row["Child_Part"],
            "Category": row["Category"],
            "Daily_Demand": row["Daily_Demand"],
            "Net_Required": row["Net_Required"],
            "Inventory_Start_Day": row["Inventory_Start_Day"],
            "Days_Since_Last": row["Days_Since_Last"],
            "Produced_Today": 1 if produce else 0,
            "Priority_Label": 0.9 if produce else 0.3  # high if produced, low otherwise
        })
        
        if produce:
            produced_qty = qty_possible
            remaining_capacity -= produced_qty * prod_time + CHANGEOVER_HOURS
            inventory[row["Child_Part"]] = max(0, inventory.get(row["Child_Part"], 0) - produced_qty)
            last_prod_day[row["Child_Part"]] = day
            
            # Boost label for neglected strangers
            if row["Category"] == "Stranger" and row["Days_Since_Last"] > 20:
                sim_rows[-1]["Priority_Label"] = 0.95

df_train = pd.DataFrame(sim_rows)
print(f"\nTraining data generated: {len(df_train):,} rows ({SIM_DAYS} days × {len(df_ml)} parts)")

print("\nCategory breakdown in training data:")
print(df_train["Category"].value_counts())

print("\nSample of training data (first 15 rows):")
print(df_train.head(15))

# Save
df_train.to_csv("120t_ml_training_data_full.csv", index=False)
print("Full training data saved → 120t_ml_training_data_full.csv")